# Job Task: Update Serving Endpoint

> **This notebook runs as a Databricks Job task.**

Task 4 (final) of the retrain pipeline:
- Inspect the current endpoint state
- **Fresh endpoint (no prior model):** deploy new champion at 100% traffic
- **Existing endpoint:** deploy new champion as a 10% canary alongside the current stable model (90/10 split)

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "", "Schema (leave blank to derive from current user)")
dbutils.widgets.text("endpoint_name", "churn_endpoint", "Endpoint Name")

catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
endpoint_name = dbutils.widgets.get("endpoint_name")

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
    TrafficConfig,
    Route,
)

catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
endpoint_name = dbutils.widgets.get("endpoint_name")

# Derive schema from the running user when not explicitly provided
if not schema:
    import re as _re
    _user = spark.sql("SELECT current_user()").first()[0]
    schema = _re.sub(r'[^a-z0-9]', '_', _user.split('@')[0].lower()).strip('_')
    print(f"schema derived from current_user: {schema}")

# Derive endpoint name from the running user when not explicitly provided
if endpoint_name.lower() == "churn_":
    import re as _re
    _user = spark.sql("SELECT current_user()").first()[0]
    endpoint_name = "churn_" + _re.sub(r'[^a-z0-9]', '_', _user.split('@')[0].lower()).strip('_')
    print(f"endpoint name derived from current_user: churn_{schema}")


new_version = dbutils.jobs.taskValues.get(taskKey="gate_and_register", key="new_version")
model_name  = f"{catalog}.{schema}.churn_classifier"

w = WorkspaceClient()

# ── Step 1: Detect current endpoint state ──────────────────────────────────
current_version = None
endpoint_exists = False
try:
    endpoint = w.serving_endpoints.get(name=endpoint_name)
    endpoint_exists = True
    served = endpoint.config.served_models if endpoint.config else []
    if served:
        routes = (
            endpoint.config.traffic_config.routes
            if endpoint.config and endpoint.config.traffic_config
            else []
        )
        if routes:
            # Find the model currently receiving the most traffic (the stable champion)
            top_route = max(routes, key=lambda r: r.traffic_percentage)
            route_to_version = {sm.name: sm.model_version for sm in served}
            current_version = route_to_version.get(top_route.served_model_name)
        else:
            current_version = served[0].model_version
except Exception:
    endpoint_exists = False

print(f"Endpoint exists  : {endpoint_exists}")
print(f"Current version  : {current_version or '(none)'}")
print(f"New champion     : v{new_version}")

# ── Step 2: Deploy ─────────────────────────────────────────────────────────
if str(current_version) == str(new_version):
    print(f"\n⚠️  Endpoint already serves v{new_version} — no update needed.")

elif current_version is None:
    # Fresh endpoint — deploy 100% to new champion
    print(f"\nFresh endpoint — deploying v{new_version} at 100% traffic")
    served_models = [ServedModelInput(
        model_name=model_name,
        model_version=new_version,
        name=f"v{new_version}",
        workload_size="Small",
        scale_to_zero_enabled=True,
    )]
    if endpoint_exists:
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_models=served_models,
        )
    else:
        w.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                name=endpoint_name,
                served_models=served_models),
        )
    print(f"✓ Deployed: 100% → v{new_version}")

else:
    # Existing model detected — canary rollout at 90/10
    print(f"\nExisting endpoint — deploying v{new_version} as 10% canary")
    served_models = [
        ServedModelInput(
            model_name=model_name,
            model_version=current_version,
            name=f"v{current_version}",
            workload_size="Small",
            scale_to_zero_enabled=True,
        ),
        ServedModelInput(
            model_name=model_name,
            model_version=new_version,
            name=f"v{new_version}",
            workload_size="Small",
            scale_to_zero_enabled=True,
        ),
    ]
    traffic_config = TrafficConfig(routes=[
        Route(served_model_name=f"v{current_version}", traffic_percentage=90),
        Route(served_model_name=f"v{new_version}",     traffic_percentage=10),
    ])
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_models=served_models,
        traffic_config=traffic_config,
    )
    print(f"✓ Canary live: 90% → v{current_version},  10% → v{new_version}")
    print(f"  Monitor metrics, then promote v{new_version} to 100% when healthy.")